In [1]:
import csv
from sage.all import is_prime, is_squarefree
import ipywidgets as widgets
from IPython.display import display, clear_output

def generate_energiedoku_csv(limit, filename="energiedoku_viertupel.csv", progress_callback=None):
    """
    Ermittelt Zahlen bis 'limit', die als Norm von quaternionischen 
    Primfaktoren (Viertupel e, a, b, c) fungieren.
    """
    results = []
    
    # Wir iterieren durch die natürlichen Zahlen und prüfen auf Quadratfreiheit
    # Im Kontext der #Energiedoku suchen wir oft nach Primzahlen, 
    # die sich als e^2 + a^2 + b^2 + c^2 darstellen lassen.
    
    for n in range(1, limit + 1):
        if is_squarefree(n):
            # In der quaternionischen Zahlentheorie (Lagrange) lässt sich jede 
            # natürliche Zahl als Summe von 4 Quadraten darstellen.
            # Wir extrahieren hier beispielhaft eine Zerlegung für das Viertupel.
            # Für eine vollständige Liste aller Zerlegungen wäre der Rechenaufwand hoch.
            
            # Beispielhafte Zerlegung (e, a, b, c) finden
            dict_decomp = four_number_sum(n)
            if dict_decomp:
                e, a, b, c = dict_decomp
                results.append([n, e, a, b, c])
        
        # Zwischenspeichern alle 100.000 Schritte für Effizienz
        if n % 100000 == 0:
            msg = f"Fortschritt: {n} Zahlen geprüft..."
            print(msg)
            if progress_callback:
                progress_callback(n, limit)

    # Speichern in CSV
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['n', 'e_family', 'a_family', 'b_family', 'c_family'])
        writer.writerows(results)
    
    print(f"Datei {filename} erfolgreich erstellt. {len(results)} Einträge gefunden.")

def four_number_sum(n):
    """Hilfsfunktion zur Zerlegung in 4 Quadrate (e^2 + a^2 + b^2 + c^2 = n)"""
    # Sage's native Methode für Summe von 4 Quadraten
    return_val = None
    try:
        # returns a list of 4 integers
        res = n.is_sum_of_four_squares(ext=True)
        return res
    except:
        # Fallback für Standard-Python/Sage-Objekte
        from sage.all import Integer
        res = Integer(n).is_sum_of_four_squares(ext=True)
        return res

# Interaktive Widgets erstellen
limit_input = widgets.BoundedIntText(
    value=1000000,
    min=1000,
    max=10**8,
    step=10000,
    description='Limit:',
    style={'description_width': 'initial'}
)

filename_input = widgets.Text(
    value='energiedoku_viertupel.csv',
    description='Dateiname:',
    style={'description_width': 'initial'}
)

run_button = widgets.Button(
    description='CSV generieren',
    button_style='success',
    icon='play'
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Fortschritt:',
    bar_style='info',
    style={'bar_color': '#00ff00'},
    layout=widgets.Layout(width='100%')
)

output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        clear_output(wait=True)
        limit = limit_input.value
        filename = filename_input.value
        
        # Progress bar zurücksetzen
        progress_bar.max = limit
        progress_bar.value = 0
        
        def update_progress(current, total):
            progress_bar.value = current
        
        print(f"Starte Berechnung bis {limit}...")
        print(f"Ergebnis wird in '{filename}' gespeichert.\n")
        
        try:
            generate_energiedoku_csv(limit, filename, update_progress)
            progress_bar.value = limit
            print("\n✓ Berechnung abgeschlossen!")
        except Exception as e:
            print(f"\n✗ Fehler: {e}")

run_button.on_click(on_button_click)

# Widgets anzeigen
controls = widgets.VBox([
    widgets.HBox([limit_input, filename_input]),
    run_button,
    progress_bar,
    output_area
])

display(controls)